# 麦轮横移诊断 / Mecanum strafe test

点击下面的 cell 触发小车动作,每次约 2 秒,结束自动刹车。
指令经 ssh 灌入板子 `/cmd_vel_joy`(手柄通道,mux 最高优先级)。

**先跑一次第 1 个 cell 定义函数,之后想动哪下点哪下。**

背景:横移指令的固件轮速图案已被编码器证实是教科书正确的
(左前−/左后+/右前+/右后−),但车体表现为原地旋转 → 怀疑麦轮左右旋
装反(俯视辊子应组成 X 形)。换轮前后各点一遍即可对比。

In [ ]:
import subprocess

BOARD = "root@192.168.13.187"

PULSE = r'''
import rclpy, time, math, sys
from rclpy.node import Node
from geometry_msgs.msg import Twist
from nav_msgs.msg import Odometry

vx, vy, wz, dur = [float(a) for a in sys.argv[1:5]]
rclpy.init()
n = Node("nb_pulse")
odom = []
n.create_subscription(Odometry, "/odom",
    lambda m: odom.append((m.pose.pose.position.x, m.pose.pose.position.y,
                           m.pose.pose.orientation.z, m.pose.pose.orientation.w)), 10)
pub = n.create_publisher(Twist, "/cmd_vel_joy", 10)
t0 = time.time()          # warm up DDS discovery so odom gets captured
while not odom and time.time() - t0 < 2.0:
    rclpy.spin_once(n, timeout_sec=0.05)
t = Twist(); t.linear.x, t.linear.y, t.angular.z = vx, vy, wz
t0 = time.time()
while time.time() - t0 < dur:
    pub.publish(t); rclpy.spin_once(n, timeout_sec=0.05)
for _ in range(3): pub.publish(Twist())
t1 = time.time()
while time.time() - t1 < 0.8:
    rclpy.spin_once(n, timeout_sec=0.05)
if len(odom) > 2:
    x0, y0, z0, w0 = odom[0]; x1, y1, z1, w1 = odom[-1]
    dyaw = math.degrees(2*math.atan2(z1, w1) - 2*math.atan2(z0, w0))
    print(f"odom: dx={x1-x0:+.3f}m dy={y1-y0:+.3f}m dyaw={dyaw:+.1f}deg")
else:
    print("no odom (nav-bringup running?)")
'''

def pulse(vx=0.0, vy=0.0, wz=0.0, dur=2.0):
    """Send (vx, vy, wz) for dur seconds via /cmd_vel_joy, then brake."""
    cmd = ("source /opt/tros/humble/setup.bash; export ROS_DOMAIN_ID=99; "
           f"python3 /dev/stdin {vx} {vy} {wz} {dur}")
    r = subprocess.run(["ssh", BOARD, cmd], input=PULSE, text=True,
                       capture_output=True, timeout=30)
    print(r.stdout.strip() or r.stderr.strip())

print("ready — 点下面的 cell 触发动作")

In [ ]:
# ⬅️ 左横移 strafe LEFT (vy=+0.2, 2s)
pulse(vy=0.2)

In [ ]:
# ➡️ 右横移 strafe RIGHT (vy=-0.2, 2s)
pulse(vy=-0.2)

In [ ]:
# ⬆️ 前进 forward (vx=+0.2, 2s) — 对照组,应该直走
pulse(vx=0.2)

In [ ]:
# 🔄 原地左旋 spin CCW (wz=+0.8, 2s) — 对照组,应该原地转
pulse(wz=0.8)

In [ ]:
# 🛑 急停 emergency stop
pulse(0.0, 0.0, 0.0, 0.3)

## 判读 / How to read

| 点的 cell | 正常表现 | 当前(轮子装反时) |
|---|---|---|
| 左横移 | 车头不变,整车向左平移,`dyaw≈0` | 原地转,`dyaw` 很大 |
| 右横移 | 向右平移,方向相反 | 原地反向转 |
| 前进 | 直走 `dx>0, dyaw≈0` | 正常(前进不受辊子方向影响) |
| 左旋 | 原地转 `dyaw>0` | 正常 |

换完轮子重新点 左横移/右横移,`dyaw` 接近 0、`dy` 明显非零即修复成功。